# Week 2 Lab — Support Ticket Triage Service (walkthrough)

Work through this alongside `src/triage/`. Each section ends with a `pytest` you should be able
to make green.

> Setup: from `projects/practice/`: `pip install -e common`, then in this folder
> `pip install -r requirements.txt`. Add keys to `projects/practice/.env` for the live parts.

In [ ]:
import sys, subprocess, json, pathlib
sys.path.insert(0, "src")
from llmlab import settings, FakeLLM
from triage.schemas import Ticket
from triage.prompts import ZeroShotStrategy, FewShotStrategy, ChainOfThoughtStrategy
print("anthropic key:", bool(settings.anthropic_api_key), "| openai key:", bool(settings.openai_api_key))

## 1 — The prompt strategies

Implement `FewShotStrategy.build` and `ChainOfThoughtStrategy.build` in `src/triage/prompts.py`, then:

In [ ]:
t = Ticket(id="demo", subject="Password reset link expired",
           body="Every link says expired. Board demo in 30 min!!", customer_tier="enterprise")
for S in (ZeroShotStrategy, FewShotStrategy, ChainOfThoughtStrategy):
    try:
        sysp, msgs = S().build(t)
        print(f"{S.__name__:24s} system={len(sysp)} chars, {len(msgs)} messages")
    except NotImplementedError as e:
        print(f"{S.__name__:24s} TODO: {e}")
# then:  !pytest -q tests/test_prompts.py

## 2 — The service: classify + rules

Implement `_classify`, `_apply_rules`, `triage`, `triage_batch` in `src/triage/service.py`. Test the rules with a `FakeLLM` (no spend):

In [ ]:
from triage.service import TriageService
fake = FakeLLM(responder=lambda m, **k: {
    "category": "security", "priority": "P4", "sentiment": "negative",
    "needs_human": False, "draft_reply": "We're investigating.", "confidence": 0.9})
try:
    r = TriageService(fake).triage(t)
    print(r.model_dump_json(indent=2))
except NotImplementedError as e:
    print("TODO:", e)
# then:  !pytest -q tests/test_service.py tests/test_schemas.py

## 3 — Evaluation + the live run

Implement `TriageEval.score_one` and `.run`. Then, with keys set and `LLM_LIVE=1`, run the real thing:

In [ ]:
# offline first (FakeLLM):  !pytest -q tests/test_evaluate.py
# then real:                !LLM_LIVE=1 pytest -q -m live -s
#
# to explore interactively once _classify works:
if settings.anthropic_api_key:
    from llmlab import get_client
    svc = TriageService(get_client("anthropic"), FewShotStrategy())
    try:
        res = svc.triage(t)
        print(res.model_dump_json(indent=2))
    except NotImplementedError as e:
        print("implement the service first:", e)
else:
    print("set ANTHROPIC_API_KEY in projects/practice/.env to run this live")

## Done when

- `pytest -q` is green (16 offline tests).
- `LLM_LIVE=1 pytest -q -m live` clears the accuracy + cost bars for **both** providers.
- You can explain why the routing and SLA logic lives in `_apply_rules` and not in the prompt.